In [ ]:
!pip install medmnist scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from medmnist import PathMNIST
from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(0)
np.random.seed(0)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SIZE, BATCH, EPOCHS, LR = 64, 128, 5, 1e-3

norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.ToTensor(), norm])
eval_tf = T.Compose([T.ToTensor(), norm])

train_ds = PathMNIST(split="train", transform=train_tf, download=True, size=SIZE)
val_ds = PathMNIST(split="val", transform=eval_tf, download=True, size=SIZE)
test_ds = PathMNIST(split="test", transform=eval_tf, download=True, size=SIZE)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH, num_workers=2)

n_classes = len(train_ds.info["label"])
class_names = [train_ds.info["label"][str(i)] for i in range(n_classes)]

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, n_classes)
model = model.to(DEV)

opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()


def evaluate(dl):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in dl:
            out = model(x.to(DEV))
            preds.append(out.argmax(1).cpu())
            labels.append(y.squeeze(1))
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


best_val, best_state = 0.0, None
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for x, y in train_dl:
        x, y = x.to(DEV), y.squeeze(1).long().to(DEV)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        total += loss.item() * len(x)
    p, l = evaluate(val_dl)
    val_acc = (p == l).mean()
    print(f"epoch {epoch+1}: train loss {total/len(train_ds):.4f}, val acc {val_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

# Final test result uses the checkpoint chosen on the validation set only
model.load_state_dict(best_state)
p, l = evaluate(test_dl)
print(f"\nTEST accuracy: {(p == l).mean():.4f}")
print("Majority-class baseline:", np.bincount(l).max() / len(l))
print(classification_report(l, p, target_names=class_names, digits=3))
print("Confusion matrix:\n", confusion_matrix(l, p))

100%|██████████| 1.07G/1.07G [04:59<00:00, 3.57MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 156MB/s]


epoch 1: train loss 0.2161, val acc 0.9407
epoch 2: train loss 0.1204, val acc 0.9715
epoch 3: train loss 0.0936, val acc 0.9145
epoch 4: train loss 0.0988, val acc 0.9668
epoch 5: train loss 0.0767, val acc 0.9775

TEST accuracy: 0.9305
Majority-class baseline: 0.18635097493036212
                                      precision    recall  f1-score   support

                             adipose      0.948     0.960     0.954      1338
                          background      0.979     1.000     0.989       847
                              debris      0.720     0.947     0.818       339
                         lymphocytes      0.985     0.957     0.971       634
                               mucus      0.974     0.909     0.941      1035
                       smooth muscle      0.771     0.927     0.842       592
                 normal colon mucosa      0.944     0.970     0.957       741
            cancer-associated stroma      0.924     0.577     0.711       421
colorectal ade